# Competitive Insights

This notebook is the first analytical layer on top of the DuckDB warehouse.

It now starts with two visual explanations:

- how the repo moves from raw scraping output to analysis-ready data
- how the warehouse tables relate to each other

After the diagrams, the notebook keeps the analytical cuts by `platform + brand`,
`zone + platform`, and `product term + platform`.


## How The Diagrams Render In Notebook

There are a few valid ways to show Mermaid-like diagrams in Jupyter environments:

- native Mermaid markdown rendering in JupyterLab-compatible frontends
- Python packages that render Mermaid diagrams through external services
- inline HTML/Javascript rendering inside notebook outputs

This notebook uses the third option as the active renderer so the diagrams can appear
inline in notebook frontends that support HTML output. The diagram definitions themselves
still use Mermaid syntax.


In [1]:

from html import escape
from pathlib import Path
from uuid import uuid4

import duckdb
import pandas as pd

try:
    from IPython.display import HTML, display
    IPYTHON_DISPLAY = True
except ImportError:
    IPYTHON_DISPLAY = False

    class HTML(str):
        pass

    def display(*args, **kwargs):
        return None

try:
    import plotly.express as px
except ImportError:
    class _DummyFigure:
        def update_layout(self, *args, **kwargs):
            return self

        def show(self, *args, **kwargs):
            return None

    class _DummyPX:
        def imshow(self, *args, **kwargs):
            return _DummyFigure()

        def bar(self, *args, **kwargs):
            return _DummyFigure()

    px = _DummyPX()

try:
    import nbformat  # noqa: F401
    NBFORMAT_AVAILABLE = True
except ImportError:
    NBFORMAT_AVAILABLE = False

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "warehouse" / "intel_rappi.duckdb").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root with data/warehouse/intel_rappi.duckdb")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DB_PATH = REPO_ROOT / "data" / "warehouse" / "intel_rappi.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)


def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()


def render_mermaid(graph_definition: str, *, height: int = 600) -> None:
    """Render Mermaid syntax inline in notebook frontends that allow HTML output."""
    if not IPYTHON_DISPLAY:
        print(graph_definition)
        return

    element_id = f"mermaid-{uuid4().hex}"
    html = f"""
    <div id="{element_id}" style="min-height:{height}px; overflow:auto; border:1px solid #e5e7eb; border-radius:12px; padding:12px; background:#fcfcfd;">
      <pre class="mermaid">{escape(graph_definition)}</pre>
    </div>
    <script>
    (function() {{
      const container = document.getElementById("{element_id}");
      if (!container) return;

      function boot() {{
        if (!window.mermaid) return;
        window.mermaid.initialize({{
          startOnLoad: false,
          theme: "neutral",
          securityLevel: "loose"
        }});
        const nodes = container.querySelectorAll(".mermaid");
        if (nodes.length > 0) {{
          window.mermaid.init(undefined, nodes);
        }}
      }}

      if (!window.mermaid) {{
        const script = document.createElement("script");
        script.src = "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.min.js";
        script.onload = boot;
        document.head.appendChild(script);
      }} else {{
        boot();
      }}
    }})();
    </script>
    """
    display(HTML(html))


print(f"Using warehouse: {DB_PATH}")


Using warehouse: /Users/jumafe/Desktop/personal_things/intel-rappi/data/warehouse/intel_rappi.duckdb


## 1. Workflow: From Scraping To Analysis-Ready Data

This diagram explains the end-to-end workflow used in the repo to transform raw platform
outputs into clean analytical tables.


In [2]:
workflow_mermaid = r"""
flowchart TB
    subgraph SCRAPE[1. Raw Collection]
        A[scripts/run_rappi_scraper.py] --> A1[rappi_<run_id>.json]
        B[scripts/run_uber_scraper.py] --> B1[uber_eats_<run_id>.json]
        C[scripts/run_didi_scraper.py] --> C1[didi_food_<run_id>.json]
    end

    subgraph RAW[2. Raw Landing Zone]
        D[data/raw/*.json]
    end

    subgraph BUILD[3. Warehouse Build]
        E[scripts/build_warehouse.py]
        E1[discover latest raw file per platform]
        E2[validate top-level structure]
        E3[validate snapshot-level fields]
        E4[normalize types and zone categories]
        E5[generate snapshot_id and derived flags]
        E6[explode products_matched into product_matches rows]
        E7[run quality checks]
    end

    subgraph WAREHOUSE[4. Analytical Storage]
        F[(intel_rappi.duckdb)]
        F1[runs]
        F2[snapshots]
        F3[product_matches]
        F4[zones_dim]
        F5[brands_dim]
        F6[data_quality_checks]
        F7[parquet exports]
    end

    subgraph ANALYSIS[5. Analysis Layer]
        G[notebooks/competitive_insights.ipynb]
        G1[warehouse inventory]
        G2[quality and metric readiness]
        G3[platform + brand cut]
        G4[zone + platform cut]
        G5[product term + platform cut]
        G6[evidence-backed insight tables]
    end

    A1 --> D
    B1 --> D
    C1 --> D
    D --> E
    E --> E1 --> E2 --> E3 --> E4 --> E5 --> E6 --> E7 --> F
    F --> F1
    F --> F2
    F --> F3
    F --> F4
    F --> F5
    F --> F6
    F --> F7
    F --> G
    G --> G1
    G --> G2
    G --> G3
    G --> G4
    G --> G5
    G --> G6
"""

render_mermaid(workflow_mermaid, height=900)


### Workflow Notes

- The 3 platform scripts produce comparable raw JSON files, but they are still operational outputs.
- `build_warehouse.py` is the cleaning and normalization step.
- The warehouse is where raw observations become structured facts, dimensions, and quality checks.
- The notebook should always read from DuckDB, not directly from the raw JSON files.


## 2. Warehouse Table Relationships

This diagram focuses on table relationships and the key analytical columns.
The full column dictionary appears immediately below.


In [3]:
schema_mermaid = r"""
erDiagram
    RUNS {
        string platform PK
        string run_id UK
        timestamp started_at
        timestamp completed_at
        bigint total_snapshots
        bigint successful
        bigint failed
        string raw_file
    }

    SNAPSHOTS {
        string snapshot_id PK
        string run_id FK
        string platform FK
        string brand_key FK
        string zone_name FK
        string restaurant_brand
        string store_id
        string store_name
        double delivery_fee
        double service_fee_pct
        double eta_minutes
        double rating
        bigint discount_count
        bigint products_total
        bigint matched_items_total
        boolean is_successful
        boolean has_product_price
        boolean has_delivery_fee
        boolean has_service_fee
        boolean has_eta
        boolean has_discount_signal
        boolean has_availability
        boolean has_final_total
    }

    PRODUCT_MATCHES {
        string snapshot_id FK
        string run_id FK
        string platform
        string brand_key FK
        string restaurant_brand
        string zone_name
        string search_term
        bigint match_rank
        string product_name
        string section
        double price
        string currency
    }

    ZONES_DIM {
        string zone_name PK
        string zone_category
        string zone_category_raw
        double zone_lat
        double zone_lng
        string zone_address
    }

    BRANDS_DIM {
        string brand_key PK
        string restaurant_brand
        string platform_id_rappi
        string platform_id_uber_eats
        string platform_id_didi_food
        bigint tracked_products_count
        bigint tracked_search_terms_count
    }

    DATA_QUALITY_CHECKS {
        string platform FK
        string run_id FK
        string check_name
        string severity
        string status
        double observed_value
        double expected_value
        string details
    }

    RUNS ||--o{ SNAPSHOTS : contains
    RUNS ||--o{ DATA_QUALITY_CHECKS : audits
    SNAPSHOTS ||--o{ PRODUCT_MATCHES : expands_to
    ZONES_DIM ||--o{ SNAPSHOTS : contextualizes
    BRANDS_DIM ||--o{ SNAPSHOTS : standardizes
    BRANDS_DIM ||--o{ PRODUCT_MATCHES : labels
"""

render_mermaid(schema_mermaid, height=1100)


## 3. Comparative Scope And Metric Coverage

This section defines what can actually be benchmarked with the current warehouse.

- `Rappi vs Uber Eats`: pricing, ETA, service-fee visibility, promo intensity, availability
- `Rappi vs DiDi Food`: pricing, promo intensity, availability
- `Rappi vs Uber Eats vs DiDi Food`: strongest on comparable product pricing and visible promotion signals

The goal is to keep the notebook decision-oriented: compare only where the current data supports a fair comparison, and call out the blind spots explicitly.


In [ ]:
CANONICAL_TERM_SQL = """
case
    when lower(search_term) like 'cuarto%' then 'Cuarto de Libra'
    else search_term
end
"""

PLATFORM_ORDER = ["rappi", "uber_eats", "didi_food"]
PLATFORM_LABELS = {
    "rappi": "Rappi",
    "uber_eats": "Uber Eats",
    "didi_food": "DiDi Food",
}
PEER_LABELS = {
    "uber_eats": "Uber Eats",
    "didi_food": "DiDi Food",
}
COVERAGE_COLUMNS = [
    ("price_cov_pct", "Product Price"),
    ("delivery_cov_pct", "Delivery Fee"),
    ("service_cov_pct", "Service Fee"),
    ("eta_cov_pct", "ETA"),
    ("discount_cov_pct", "Discount Signal"),
    ("availability_cov_pct", "Availability"),
]

def pretty_platform(value: str) -> str:
    return PLATFORM_LABELS.get(value, value)

def pretty_peer(value: str) -> str:
    return PEER_LABELS.get(value, value)

def show_frame(frame: pd.DataFrame) -> None:
    if IPYTHON_DISPLAY:
        display(frame)
    else:
        print(frame.to_string(index=False))

def show_note(title: str, bullets: list[str]) -> None:
    if IPYTHON_DISPLAY:
        items = "".join(f"<li>{escape(item)}</li>" for item in bullets)
        html = (
            "<div style=\"border:1px solid #e5e7eb; border-radius:12px; padding:14px; margin:12px 0; background:#fcfcfd;\">"
            f"<strong>{escape(title)}</strong><ul>{items}</ul></div>"
        )
        display(HTML(html))
    else:
        print(title)
        for item in bullets:
            print(f"- {item}")

PLOTLY_RENDER_WARNING_SHOWN = False

def show_figure(fig) -> None:
    global PLOTLY_RENDER_WARNING_SHOWN
    if not IPYTHON_DISPLAY:
        return
    if not NBFORMAT_AVAILABLE:
        if not PLOTLY_RENDER_WARNING_SHOWN:
            show_note(
                "Plotly rendering dependency missing",
                [
                    "Rich chart rendering in notebook outputs needs `nbformat>=4.2.0` in the active kernel.",
                    "Run `pip install -r requirements.txt -r requirements-dev.txt` or `pip install nbformat`, then restart the kernel.",
                ],
            )
            PLOTLY_RENDER_WARNING_SHOWN = True
        return
    try:
        fig.show()
    except ValueError as exc:
        if "Mime type rendering requires nbformat>=4.2.0" not in str(exc):
            raise
        if not PLOTLY_RENDER_WARNING_SHOWN:
            show_note(
                "Plotly rendering dependency missing",
                [
                    "Rich chart rendering in notebook outputs needs `nbformat>=4.2.0` in the active kernel.",
                    "Run `pip install -r requirements.txt -r requirements-dev.txt` or `pip install nbformat`, then restart the kernel.",
                ],
            )
            PLOTLY_RENDER_WARNING_SHOWN = True

def join_values(values: list[str]) -> str:
    cleaned = [str(value) for value in values if pd.notna(value) and str(value).strip()]
    cleaned = list(dict.fromkeys(cleaned))
    if not cleaned:
        return "none"
    if len(cleaned) == 1:
        return cleaned[0]
    return ", ".join(cleaned[:-1]) + f" and {cleaned[-1]}"

def classify_position(avg_pct_gap: float, threshold: float = 5.0) -> str:
    if pd.isna(avg_pct_gap):
        return "No data"
    if avg_pct_gap <= -threshold:
        return "Rappi cheaper"
    if avg_pct_gap >= threshold:
        return "Rappi pricier"
    return "Similar"

metric_coverage = q("""
select platform,
       count(*) as snapshots,
       round(avg(case when has_product_price then 1 else 0 end) * 100, 1) as price_cov_pct,
       round(avg(case when has_delivery_fee then 1 else 0 end) * 100, 1) as delivery_cov_pct,
       round(avg(case when has_service_fee then 1 else 0 end) * 100, 1) as service_cov_pct,
       round(avg(case when has_eta then 1 else 0 end) * 100, 1) as eta_cov_pct,
       round(avg(case when has_discount_signal then 1 else 0 end) * 100, 1) as discount_cov_pct,
       round(avg(case when has_availability then 1 else 0 end) * 100, 1) as availability_cov_pct
from snapshots
group by 1
order by 1
""")

metric_coverage["metrics_with_observed_coverage"] = metric_coverage[[name for name, _ in COVERAGE_COLUMNS]].gt(0).sum(axis=1)
coverage_display = metric_coverage.copy()
coverage_display["platform"] = coverage_display["platform"].map(pretty_platform)
show_frame(coverage_display)

coverage_heatmap = metric_coverage.set_index("platform")[[name for name, _ in COVERAGE_COLUMNS]]
coverage_heatmap.index = [pretty_platform(value) for value in coverage_heatmap.index]
coverage_heatmap.columns = [label for _, label in COVERAGE_COLUMNS]
coverage_fig = px.imshow(coverage_heatmap, aspect="auto", color_continuous_scale="Blues", labels={"color": "Coverage %"})
coverage_fig.update_layout(title="Observed Metric Coverage by Platform (%)", height=380)
show_figure(coverage_fig)

scope_bullets = []
for _, row in metric_coverage.iterrows():
    observed = [label for column, label in COVERAGE_COLUMNS if row[column] > 0]
    scope_bullets.append(
        f"{pretty_platform(row['platform'])} has observed coverage for {row['metrics_with_observed_coverage']}/6 core metrics: {join_values(observed)}."
    )
show_note("Scope from the current warehouse", scope_bullets)


## 4. Price Positioning

To avoid unfair comparisons, prices are benchmarked at the most comparable grain available in the warehouse: `brand + zone + normalized tracked product term`.

Because each scraper can return more than one matched item per term, the notebook uses the **lowest observed matched price per platform/zone/term** as the closest customer-facing comparable offer.


In [ ]:
product_term = q(f"""
with canonical as (
    select platform, restaurant_brand, zone_name,
           {CANONICAL_TERM_SQL} as canonical_term,
           min(price) as snapshot_price
    from product_matches
    group by 1, 2, 3, 4
), median_prices as (
    select platform, restaurant_brand, canonical_term,
           count(*) as comparable_snapshots,
           round(avg(snapshot_price), 2) as avg_snapshot_price,
           round(median(snapshot_price), 2) as median_snapshot_price
    from canonical
    group by 1, 2, 3
)
select *
from median_prices
order by restaurant_brand, canonical_term, platform
""")

product_term["platform_label"] = product_term["platform"].map(pretty_platform)
price_matrix = product_term.pivot_table(
    index=["restaurant_brand", "canonical_term"],
    columns="platform_label",
    values="median_snapshot_price",
    aggfunc="first",
)
price_matrix = price_matrix.reindex(columns=["Rappi", "Uber Eats", "DiDi Food"])
price_matrix.index = [f"{brand} | {term}" for brand, term in price_matrix.index]
show_frame(price_matrix.reset_index().rename(columns={"index": "brand_term"}))

price_fig = px.imshow(price_matrix, aspect="auto", color_continuous_scale="Oranges", labels={"color": "Median comparable price (MXN)"})
price_fig.update_layout(title="Median Comparable Price by Brand / Product Term", height=520)
show_figure(price_fig)

platform_brand = q(f"""
with canonical as (
    select platform, restaurant_brand, zone_name,
           {CANONICAL_TERM_SQL} as canonical_term,
           min(price) as snapshot_price
    from product_matches
    group by 1, 2, 3, 4
), pairwise as (
    select 'uber_eats' as peer, r.restaurant_brand, r.zone_name, r.canonical_term,
           r.snapshot_price as rappi_price, u.snapshot_price as peer_price,
           r.snapshot_price - u.snapshot_price as gap_mxn,
           100.0 * (r.snapshot_price - u.snapshot_price) / nullif(u.snapshot_price, 0) as pct_gap
    from canonical r
    join canonical u using (restaurant_brand, zone_name, canonical_term)
    where r.platform = 'rappi' and u.platform = 'uber_eats'
    union all
    select 'didi_food' as peer, r.restaurant_brand, r.zone_name, r.canonical_term,
           r.snapshot_price as rappi_price, d.snapshot_price as peer_price,
           r.snapshot_price - d.snapshot_price as gap_mxn,
           100.0 * (r.snapshot_price - d.snapshot_price) / nullif(d.snapshot_price, 0) as pct_gap
    from canonical r
    join canonical d using (restaurant_brand, zone_name, canonical_term)
    where r.platform = 'rappi' and d.platform = 'didi_food'
)
select peer, restaurant_brand, canonical_term,
       count(*) as comparisons,
       round(avg(gap_mxn), 2) as avg_gap_mxn,
       round(median(gap_mxn), 2) as median_gap_mxn,
       round(avg(pct_gap), 2) as avg_pct_gap,
       round(median(pct_gap), 2) as median_pct_gap
from pairwise
group by 1, 2, 3
order by 1, 2, 3
""")

platform_brand["peer_label"] = platform_brand["peer"].map(pretty_peer)
platform_brand["positioning"] = platform_brand["avg_pct_gap"].apply(classify_position)
price_summary_display = platform_brand[[
    "peer_label", "restaurant_brand", "canonical_term", "comparisons",
    "avg_gap_mxn", "avg_pct_gap", "positioning"
]].rename(columns={"peer_label": "peer"})
show_frame(price_summary_display)

price_rank = product_term.copy()
price_rank["price_rank"] = price_rank.groupby(["restaurant_brand", "canonical_term"])["median_snapshot_price"].rank(method="dense")
cheapest_or_tied = price_rank[price_rank["price_rank"] == 1].groupby("platform_label").size().to_dict()

uber_terms = platform_brand[platform_brand["peer"] == "uber_eats"]
didi_terms = platform_brand[platform_brand["peer"] == "didi_food"]
show_note(
    "Pricing takeaways",
    [
        f"Against Uber Eats, Rappi is pricier on {join_values(uber_terms.loc[uber_terms['positioning'] == 'Rappi pricier', 'canonical_term'].tolist())} and similar on {join_values(uber_terms.loc[uber_terms['positioning'] == 'Similar', 'canonical_term'].tolist())}.",
        f"Against DiDi Food, Rappi is cheaper on {join_values(didi_terms.loc[didi_terms['positioning'] == 'Rappi cheaper', 'canonical_term'].tolist())}, similar on {join_values(didi_terms.loc[didi_terms['positioning'] == 'Similar', 'canonical_term'].tolist())}, and pricier on {join_values(didi_terms.loc[didi_terms['positioning'] == 'Rappi pricier', 'canonical_term'].tolist())}.",
        f"Across the 6 tracked product families, Uber Eats is cheapest or tied-cheapest on {cheapest_or_tied.get('Uber Eats', 0)} terms, Rappi on {cheapest_or_tied.get('Rappi', 0)}, and DiDi Food on {cheapest_or_tied.get('DiDi Food', 0)}.",
    ],
)


## 5. Operational Advantage / Disadvantage

Operational benchmarking is currently strongest for `Rappi vs Uber Eats`, because DiDi Food does not expose ETA in the warehouse snapshot set.


In [ ]:
eta_summary = q("""
select platform,
       count(*) filter (where has_eta) as eta_snapshots,
       round(avg(eta_minutes), 2) as avg_eta_minutes,
       round(median(eta_minutes), 2) as median_eta_minutes
from snapshots
group by 1
order by 1
""")
eta_summary["platform_label"] = eta_summary["platform"].map(pretty_platform)
show_frame(eta_summary)

eta_fig = px.bar(
    eta_summary.dropna(subset=["median_eta_minutes"]),
    x="platform_label",
    y="median_eta_minutes",
    color="platform_label",
    labels={"platform_label": "Platform", "median_eta_minutes": "Median ETA (min)"},
)
eta_fig.update_layout(title="Median ETA by Platform", height=360, showlegend=False)
show_figure(eta_fig)

eta_zone_gap = q("""
with latest as (
    select *,
           row_number() over (partition by platform, zone_name, restaurant_brand order by scraped_at desc nulls last) as rn
    from snapshots
), filtered as (
    select *
    from latest
    where rn = 1
)
select zone_name,
       round(avg(case when platform = 'rappi' then eta_minutes end), 2) as rappi_eta,
       round(avg(case when platform = 'uber_eats' then eta_minutes end), 2) as uber_eta,
       round(avg(case when platform = 'rappi' then eta_minutes end) - avg(case when platform = 'uber_eats' then eta_minutes end), 2) as eta_gap_rappi_vs_uber
from filtered
group by 1
order by eta_gap_rappi_vs_uber desc nulls last, zone_name
""")
show_frame(eta_zone_gap.dropna(subset=["eta_gap_rappi_vs_uber"]).head(10))

top_eta_lag_zones = eta_zone_gap.dropna(subset=["eta_gap_rappi_vs_uber"]).head(3)["zone_name"].tolist()
avg_eta_gap = round(eta_zone_gap["eta_gap_rappi_vs_uber"].dropna().mean(), 2)
median_eta_gap = round(eta_zone_gap["eta_gap_rappi_vs_uber"].dropna().median(), 2)
show_note(
    "Operational readout",
    [
        f"Rappi median ETA is {eta_summary.loc[eta_summary['platform'] == 'rappi', 'median_eta_minutes'].iloc[0]} min versus {eta_summary.loc[eta_summary['platform'] == 'uber_eats', 'median_eta_minutes'].iloc[0]} min for Uber Eats.",
        f"Across zones with both platforms present, Rappi is slower than Uber Eats by {avg_eta_gap} min on average and {median_eta_gap} min on median.",
        f"The largest ETA gaps appear in {join_values(top_eta_lag_zones)}.",
        "DiDi Food currently has no ETA coverage in the warehouse, so the operational comparison is a Rappi-vs-Uber view only.",
    ],
)


## 6. Fee Structure

Fee benchmarking needs two angles: the observed fee values themselves and the share of snapshots where each fee is actually visible. That matters because delivery-fee and service-fee coverage is asymmetric across platforms.


In [ ]:
fee_summary = q("""
select platform,
       count(*) as snapshots,
       count(delivery_fee) as delivery_fee_rows,
       round(avg(case when has_delivery_fee then 1 else 0 end) * 100, 1) as delivery_fee_coverage_pct,
       round(avg(delivery_fee), 2) as avg_delivery_fee_mxn,
       round(median(delivery_fee), 2) as median_delivery_fee_mxn,
       round(max(delivery_fee), 2) as max_delivery_fee_mxn,
       count(service_fee_pct) as service_fee_rows,
       round(avg(case when has_service_fee then 1 else 0 end) * 100, 1) as service_fee_coverage_pct,
       round(avg(service_fee_pct), 2) as avg_service_fee_pct,
       round(median(service_fee_pct), 2) as median_service_fee_pct
from snapshots
group by 1
order by 1
""")
fee_summary["platform_label"] = fee_summary["platform"].map(pretty_platform)
show_frame(fee_summary)

fee_plot = pd.DataFrame([
    {"platform": row.platform_label, "metric": "Median Delivery Fee (MXN)", "median_value": row.median_delivery_fee_mxn}
    for row in fee_summary.itertuples()
] + [
    {"platform": row.platform_label, "metric": "Median Service Fee (%)", "median_value": row.median_service_fee_pct}
    for row in fee_summary.itertuples()
])
fee_fig = px.bar(
    fee_plot,
    x="platform",
    y="median_value",
    color="metric",
    barmode="group",
    labels={"platform": "Platform", "median_value": "Observed median value"},
)
fee_fig.update_layout(title="Observed Median Fees by Platform", height=380)
show_figure(fee_fig)

rappi_fee = fee_summary[fee_summary["platform"] == "rappi"].iloc[0]
show_note(
    "Fee readout",
    [
        f"Rappi is the only platform with observed delivery-fee values in this warehouse snapshot: median {rappi_fee.median_delivery_fee_mxn} MXN, average {rappi_fee.avg_delivery_fee_mxn} MXN, and a max of {rappi_fee.max_delivery_fee_mxn} MXN.",
        "Uber Eats and DiDi Food do not expose a comparable delivery-fee field in the current warehouse, so direct delivery-fee benchmarking is asymmetric.",
        f"For service fees, both Rappi and Uber Eats show a median extracted value of {fee_summary.loc[fee_summary['platform'] == 'rappi', 'median_service_fee_pct'].iloc[0]}%, which means the current dataset does not show a measurable service-fee premium between them.",
    ],
)


## 7. Promotional Strategy

The warehouse captures promo intensity directly through `discount_count`. Promo *type* is inferred here from keywords visible in matched tracked products, so it should be read as a directional pattern, not as a full promotion catalog.


In [ ]:
promo_intensity = q("""
select platform,
       round(avg(discount_count), 2) as avg_discount_count,
       round(median(discount_count), 2) as median_discount_count,
       round(avg(case when discount_count = 0 then 1 else 0 end) * 100, 1) as zero_discount_snapshot_pct
from snapshots
group by 1
order by 1
""")
promo_intensity["platform_label"] = promo_intensity["platform"].map(pretty_platform)
show_frame(promo_intensity)

promo_types = q(r"""
with matched as (
    select platform,
           lower(coalesce(product_name, '') || ' ' || coalesce(description, '')) as text
    from product_matches
), exploded as (
    select platform, 'bundle_combo' as promo_type from matched where regexp_matches(text, 'combo')
    union all
    select platform, 'freebie' as promo_type from matched where regexp_matches(text, 'gratis')
    union all
    select platform, 'gift_or_promotional_item' as promo_type from matched where regexp_matches(text, 'promocional|vaso')
    union all
    select platform, 'multipack' as promo_type from matched where regexp_matches(text, '2x1|3x2')
    union all
    select platform, 'percent_discount' as promo_type from matched where regexp_matches(text, '[0-9]+\s*%')
)
select platform, promo_type, count(*) as matched_rows
from exploded
group by 1, 2
order by 1, 3 desc, 2
""")

promo_types["platform_label"] = promo_types["platform"].map(pretty_platform)
promo_types["share_pct"] = (
    promo_types["matched_rows"]
    / promo_types.groupby("platform")["matched_rows"].transform("sum")
    * 100
).round(1)
show_frame(promo_types)

promo_fig = px.bar(
    promo_types,
    x="platform_label",
    y="share_pct",
    color="promo_type",
    barmode="stack",
    labels={"platform_label": "Platform", "share_pct": "Share of inferred promo-type signals (%)"},
)
promo_fig.update_layout(title="Inferred Promotion Mix from Matched Product Text", height=420)
show_figure(promo_fig)

rappi_discounts = promo_intensity[promo_intensity["platform"] == "rappi"]["avg_discount_count"].iloc[0]
uber_discounts = promo_intensity[promo_intensity["platform"] == "uber_eats"]["avg_discount_count"].iloc[0]
didi_discounts = promo_intensity[promo_intensity["platform"] == "didi_food"]["avg_discount_count"].iloc[0]
show_note(
    "Promotional readout",
    [
        f"Visible promo intensity is materially lower on Rappi ({rappi_discounts} average discount signals per snapshot) than on Uber Eats ({uber_discounts}) and DiDi Food ({didi_discounts}).",
        f"Across tracked matched products, Uber Eats and DiDi Food lean most heavily into {join_values(promo_types[promo_types['platform'].isin(['uber_eats', 'didi_food'])].sort_values('matched_rows', ascending=False).head(2)['promo_type'].tolist())} patterns, while Rappi shows fewer freebie signals and more limited promo breadth.",
        "Promo types here are inferred from tracked matched-product text, so they complement `discount_count` instead of replacing it as the main promo-intensity metric.",
    ],
)


## 8. Geographic Variability

Citywide averages hide where competitiveness actually breaks. This section shows the zone-by-zone variation in Rappi price gaps versus Uber Eats and DiDi Food, plus the largest ETA gaps versus Uber Eats.


In [ ]:
zone_platform = q(f"""
with canonical as (
    select platform, restaurant_brand, zone_name,
           {CANONICAL_TERM_SQL} as canonical_term,
           min(price) as snapshot_price
    from product_matches
    group by 1, 2, 3, 4
), price_gap as (
    select 'uber_eats' as peer, r.zone_name, count(*) as comparisons,
           round(avg(r.snapshot_price - u.snapshot_price), 2) as avg_gap_mxn,
           round(median(r.snapshot_price - u.snapshot_price), 2) as median_gap_mxn
    from canonical r
    join canonical u using (restaurant_brand, zone_name, canonical_term)
    where r.platform = 'rappi' and u.platform = 'uber_eats'
    group by 1, 2
    union all
    select 'didi_food' as peer, r.zone_name, count(*) as comparisons,
           round(avg(r.snapshot_price - d.snapshot_price), 2) as avg_gap_mxn,
           round(median(r.snapshot_price - d.snapshot_price), 2) as median_gap_mxn
    from canonical r
    join canonical d using (restaurant_brand, zone_name, canonical_term)
    where r.platform = 'rappi' and d.platform = 'didi_food'
    group by 1, 2
), eta_gap as (
    with latest as (
        select *,
               row_number() over (partition by platform, zone_name, restaurant_brand order by scraped_at desc nulls last) as rn
        from snapshots
    ), filtered as (
        select *
        from latest
        where rn = 1
    )
    select zone_name,
           round(avg(case when platform = 'rappi' then eta_minutes end) - avg(case when platform = 'uber_eats' then eta_minutes end), 2) as eta_gap_rappi_vs_uber
    from filtered
    group by 1
)
select p.peer, p.zone_name, p.comparisons, p.avg_gap_mxn, p.median_gap_mxn, e.eta_gap_rappi_vs_uber
from price_gap p
left join eta_gap e using (zone_name)
order by p.peer, p.zone_name
""")

zone_platform["peer_label"] = zone_platform["peer"].map(pretty_peer)
show_frame(zone_platform)

geo_matrix = zone_platform.pivot_table(index="peer_label", columns="zone_name", values="median_gap_mxn", aggfunc="first")
geo_fig = px.imshow(geo_matrix, aspect="auto", color_continuous_scale="RdBu_r", labels={"color": "Median Rappi price gap (MXN)"})
geo_fig.update_layout(title="Zone-Level Median Price Gap: Rappi vs Each Competitor", height=420)
show_figure(geo_fig)

top_uber_gap = zone_platform[zone_platform["peer"] == "uber_eats"].sort_values("median_gap_mxn", ascending=False).head(3)["zone_name"].tolist()
top_didi_gap = zone_platform[zone_platform["peer"] == "didi_food"].sort_values("median_gap_mxn", ascending=False).head(3)["zone_name"].tolist()
top_didi_advantage = zone_platform[zone_platform["peer"] == "didi_food"].sort_values("median_gap_mxn", ascending=True).head(3)["zone_name"].tolist()
top_eta_geo = zone_platform.dropna(subset=["eta_gap_rappi_vs_uber"]).sort_values("eta_gap_rappi_vs_uber", ascending=False).head(3)["zone_name"].tolist()
show_note(
    "Geographic readout",
    [
        f"Against Uber Eats, Rappi shows its largest price disadvantage in {join_values(top_uber_gap)}.",
        f"Against DiDi Food, Rappi is most expensive in {join_values(top_didi_gap)} but cheapest in {join_values(top_didi_advantage)}.",
        f"The biggest ETA disadvantages versus Uber Eats show up in {join_values(top_eta_geo)}.",
        "Competitiveness is not uniform across CDMX: zone-level gaps are large enough that one citywide average would hide both strongholds and problem pockets.",
    ],
)
